In [ ]:
!pip install category_encoders
!pip install xgboost
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,KFold,cross_val_score,GridSearchCV
from sklearn.linear_model import LinearRegression,Ridge,Lasso
import seaborn as sns
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.metrics import r2_score,mean_absolute_error
from sklearn.decomposition import  PCA
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor,RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from xgboost import XGBClassifier
import category_encoders  as ce


In [ ]:
!pip install optuna


In [ ]:
df=pd.read_csv("/content/selection").drop(columns=['pooja room',"others",'Unnamed: 0','study room','floorNum'])
df.shape


(3234, 12)

In [ ]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_score
0,flat,sector 36,0.82,3.0,2.0,2,New Property,908.0,0.0,0.0,0.0,Low
1,flat,sector 89,0.95,2.0,2.0,2,New Property,1432.0,1.0,0.0,0.0,Low
2,flat,sohna road,0.32,2.0,2.0,1,New Property,1000.0,0.0,0.0,0.0,Low
3,flat,sector 92,1.60,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,1.0,High
4,flat,sector 102,0.48,2.0,2.0,1,Relatively New,630.0,0.0,1.0,0.0,High


In [ ]:
df.columns

Index(['property_type', 'sector', 'price', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'luxury_score'],
      dtype='object')

In [ ]:
x=df.drop("price",axis=1)
y=df["price"]

In [ ]:
y_trans=np.log1p(y)

In [ ]:
columns=['property_type', 'sector', 'balcony','agePossession',
       'luxury_score','furnishing_type']

In [ ]:
def categorize(x):
  if x==0:
    return "unfurnished"
  elif x==1:
    return 'semiunfurnished'
  else:
    return 'furnished'

In [ ]:
df["furnishing_type"]=df["furnishing_type"].apply(categorize)
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3234 entries, 0 to 3233
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   property_type    3234 non-null   object 
 1   sector           3234 non-null   object 
 2   price            3234 non-null   float64
 3   bedRoom          3234 non-null   float64
 4   bathroom         3234 non-null   float64
 5   balcony          3234 non-null   object 
 6   agePossession    3234 non-null   object 
 7   built_up_area    3234 non-null   float64
 8   servant room     3234 non-null   float64
 9   store room       3234 non-null   float64
 10  furnishing_type  3234 non-null   object 
 11  luxury_score     3234 non-null   object 
dtypes: float64(6), object(6)
memory usage: 303.3+ KB


In [ ]:
 preprocessor=ColumnTransformer([('num',StandardScaler(),['bedRoom','built_up_area','servant room','store room','bathroom']),
                                ('cat',OneHotEncoder(handle_unknown='ignore',drop="first"),['agePossession']),
                                  ('order',OrdinalEncoder(),['balcony','luxury_score','property_type','furnishing_type']),('target',ce.TargetEncoder(),["sector"])
                                  ],remainder="passthrough")
pipeline=Pipeline([('prerocessor',preprocessor),('regressor',LinearRegression())])


In [ ]:
kfold=KFold(n_splits=10,shuffle=True,random_state=42)
score=cross_val_score(pipeline,x,y_trans,cv=kfold,scoring="r2")

In [ ]:
print(score.mean(),score.std())

0.7925487274111884 0.02351619814880503


In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y_trans,test_size=0.2,random_state=42)

In [ ]:
pipeline.fit(x_train,y_train)

Pipeline(steps=[('prerocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'built_up_area',
                                                   'servant room', 'store room',
                                                   'bathroom']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['agePossession']),
                                                 ('order', OrdinalEncoder(),
                                                  ['balcony', 'luxury_score',
                                                   'property_type',
                                                   'furnishing_type']),
                                                 ('target', TargetEncoder(),
                                                  ['sector'])])),
                ('regressor', LinearRegression())])

In [ ]:
y_pred=pipeline.predict(x_test)
y_pred=np.expm1(y_pred)

In [ ]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.4371342885048375

In [ ]:
def scorer(model_name,model):
  output=[]
  output.append(model_name)
  pipeline=Pipeline([('prerocessor',preprocessor),(model_name,model)])
  kfold=KFold(n_splits=10,shuffle=True,random_state=42)
  score=cross_val_score(pipeline,x,y_trans,cv=kfold,scoring="r2")
  output.append(score.mean())
  x_train,x_test,y_train,y_test=train_test_split(x,y_trans,test_size=0.2,random_state=42)
  pipeline.fit(x_train,y_train)
  y_pred=pipeline.predict(x_test)
  y_pred=np.expm1(y_pred)
  output.append(mean_absolute_error(np.expm1(y_test),y_pred))
  return output



In [ ]:
model_dict={
    "linear_model":LinearRegression(),
    'decision_tree':DecisionTreeRegressor(),
    'svm':SVR(kernel='rbf'),
    'random_forest':RandomForestRegressor(),
    'gradient_boosting':GradientBoostingRegressor(),
    'ridge':Ridge(),
    'lasso':Lasso(),
    'xgboost':XGBRegressor(),
    'mlp': MLPRegressor(),
    'extra_tree':ExtraTreesRegressor()

}

In [ ]:
model_output=[]
for model_name,model in model_dict.items():
  model_output.append(scorer(model_name,model))

In [ ]:
model_df=pd.DataFrame(model_output,columns=["model_name","r2_score","mean_score"]).sort_values("mean_score",ascending=True)


In [ ]:
model_df

,model_name,r2_score,mean_score
9,extra_tree,0.875679,0.291519
7,xgboost,0.878216,0.298130
3,random_forest,0.872464,0.298223
4,gradient_boosting,0.855341,0.335230
2,svm,0.822617,0.371275
1,decision_tree,0.767264,0.375040
8,mlp,0.812187,0.389019
0,linear_model,0.792549,0.437134
5,ridge,0.792590,0.437579
6,lasso,-0.002016,0.877895


In [ ]:
def scorer1(model_name,model):
  output1=[]
  output1.append(model_name)
  preprocessor=ColumnTransformer([('num',StandardScaler(),['bedRoom','built_up_area','servant room','store room','bathroom']),
                                ('cat',OneHotEncoder(handle_unknown='ignore',drop="first"),["sector",'agePossession']),
                                  ('order',OrdinalEncoder(),['balcony','luxury_score','property_type','furnishing_type'])
                                  ],remainder="passthrough")
  pipeline=Pipeline([('prerocessor',preprocessor),('pca',PCA(n_components=45)),(model_name,model)])
  kfold=KFold(n_splits=10,shuffle=True,random_state=42)
  score=cross_val_score(pipeline,x,y_trans,cv=kfold,scoring="r2")
  output1.append(score.mean())
  x_train,x_test,y_train,y_test=train_test_split(x,y_trans,test_size=0.2,random_state=42)
  pipeline.fit(x_train,y_train)
  y_pred=pipeline.predict(x_test)
  y_pred=np.expm1(y_pred)
  output1.append(mean_absolute_error(np.expm1(y_test),y_pred))
  return output1



In [ ]:
model_output1=[]
for model_name,model in model_dict.items():
  model_output1.append(scorer1(model_name,model))


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categ

In [ ]:
model_df1=pd.DataFrame(model_output,columns=["model_name","r2_score","mean_score"]).sort_values("mean_score",ascending=True)


In [ ]:
model_df1

,model_name,r2_score,mean_score
9,extra_tree,0.875679,0.291519
7,xgboost,0.878216,0.298130
3,random_forest,0.872464,0.298223
4,gradient_boosting,0.855341,0.335230
2,svm,0.822617,0.371275
1,decision_tree,0.767264,0.375040
8,mlp,0.812187,0.389019
0,linear_model,0.792549,0.437134
5,ridge,0.792590,0.437579
6,lasso,-0.002016,0.877895


Target_encoders


In [ ]:
def target(model_name,model):
  output1=[]
  output1.append(model_name)
  preprocessor=ColumnTransformer([('num',StandardScaler(),['bedRoom','built_up_area','servant room','store room','bathroom']),
                                ('cat',OneHotEncoder(handle_unknown='ignore',drop="first"),['agePossession']),
                                  ('order',OrdinalEncoder(),['balcony','luxury_score','property_type','furnishing_type']),('target',ce.TargetEncoder(),["sector"])
                                  ],remainder="passthrough")
  pipeline=Pipeline([('prerocessor',preprocessor),(model_name,model)])
  kfold=KFold(n_splits=10,shuffle=True,random_state=42)
  score=cross_val_score(pipeline,x,y_trans,cv=kfold,scoring="r2")
  output1.append(score.mean())
  x_train,x_test,y_train,y_test=train_test_split(x,y_trans,test_size=0.2,random_state=42)
  pipeline.fit(x_train,y_train)
  y_pred=pipeline.predict(x_test)
  y_pred=np.expm1(y_pred)
  output1.append(mean_absolute_error(np.expm1(y_test),y_pred))
  return output1



In [ ]:
target_list=[]
for model_name,model in model_dict.items():
  target_list.append(target(model_name,model))


In [ ]:
target1=pd.DataFrame(target_list,columns=["model_name","r2_score","mean_score"]).sort_values("mean_score",ascending=True)

In [ ]:
target1

,model_name,r2_score,mean_score
9,extra_tree,0.877741,0.286742
7,xgboost,0.878216,0.298130
3,random_forest,0.873766,0.300612
4,gradient_boosting,0.855361,0.334505
2,svm,0.822617,0.371275
1,decision_tree,0.766908,0.372251
8,mlp,0.813433,0.392643
0,linear_model,0.792549,0.437134
5,ridge,0.792590,0.437579
6,lasso,-0.002016,0.877895


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
num_cols = ['bedRoom', 'built_up_area', 'servant room', 'store room', 'bathroom']

cat_cols = ['sector', 'agePossession']

ord_cols = ['balcony', 'luxury_score', 'property_type', 'furnishing_type']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols),
    ('ord', OrdinalEncoder(), ord_cols)
], remainder='passthrough')
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        random_state=42,
        n_jobs=-1
    ))
])
param_grid = {
    'model__n_estimators': [100, 300],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth': [3, 6],
    'model__subsample': [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0],
    'model__reg_lambda': [1, 5],
    'model__reg_alpha': [0, 0.1]
}
grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)
grid.fit(x, y_trans)
print("Best Params:", grid.best_params_)
print("Best Score:", grid.best_score_)



Fitting 5 folds for each of 128 candidates, totalling 640 fits
Best Params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.1, 'model__max_depth': 6, 'model__n_estimators': 300, 'model__reg_alpha': 0, 'model__reg_lambda': 1, 'model__subsample': 0.8}
Best Score: 0.8826084937444767


In [ ]:

# from sklearn.pipeline import Pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
# from sklearn.model_selection import GridSearchCV
# from xgboost import XGBRegressor
# num_cols = ['bedRoom', 'built_up_area', 'servant room', 'store room', 'bathroom']

# cat_cols = ['sector', 'agePossession']

# ord_cols = ['balcony', 'luxury_score', 'property_type', 'furnishing_type']

# preprocessor = ColumnTransformer([
#     ('num', StandardScaler(), num_cols),
#     ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols),
#     ('ord', OrdinalEncoder(), ord_cols)
# ], remainder='passthrough')
# pipeline = Pipeline([
#     ('preprocessor', preprocessor),
#     ('model', SVR())
# ])
# grid = GridSearchCV(
#     pipeline,
#     param_grid=param_grid,
#     cv=5,
#     scoring='r2',
#     n_jobs=-1,
# )
# grid.fit(x, y_trans)
# print("Best Params:", grid.best_params_)
# print("Best Score:", grid.best_score_)



ValueError: Invalid parameter 'colsample_bytree' for estimator SVR(). Valid parameters are: ['C', 'cache_size', 'coef0', 'degree', 'epsilon', 'gamma', 'kernel', 'max_iter', 'shrinking', 'tol', 'verbose'].

In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
num_cols = ['bedRoom', 'built_up_area', 'servant room', 'store room', 'bathroom']

cat_cols = ['sector', 'agePossession']

ord_cols = ['balcony', 'luxury_score', 'property_type', 'furnishing_type']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols),
    ('ord', OrdinalEncoder(), ord_cols)
], remainder='passthrough')
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model',SVR(
    C=10,
    epsilon=0.1,
    gamma='auto',
    kernel='rbf'
)
)
])
pipeline.fit(x_train,y_train)
y_pred=pipeline.predict(x_test)
y_pred=np.expm1(y_pred)

print(r2_score(np.expm1(y_test),y_pred))
print(mean_absolute_error(np.expm1(y_test),y_pred))
print()
y_pred=pipeline.predict(x_train)
y_pred=np.expm1(y_pred)

print(r2_score(np.expm1(y_train),y_pred))
print(mean_absolute_error(np.expm1(y_train),y_pred))






0.8457780976304607
0.3240744440314632

0.9029768311701314
0.29756031759540863


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
num_cols = ['bedRoom', 'built_up_area', 'servant room', 'store room', 'bathroom']

cat_cols = ['sector', 'agePossession']

ord_cols = ['balcony', 'luxury_score', 'property_type', 'furnishing_type']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols),
    ('ord', OrdinalEncoder(), ord_cols)
], remainder='passthrough')
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor())
])

param_grid = {
    'model__n_estimators': [100, 300],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2],
    'model__max_features': ['sqrt', 'log2']
}
grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)
grid.fit(x, y_trans)
print("Best Params:", grid.best_params_)
print("Best Score:", grid.best_score_)



Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best Params: {'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 300}
Best Score: 0.880523286996133


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:05:45] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "max_features", "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
num_cols = ['bedRoom', 'built_up_area', 'servant room', 'store room', 'bathroom']

cat_cols = ['sector', 'agePossession']

ord_cols = ['balcony', 'luxury_score', 'property_type', 'furnishing_type']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols),
    ('ord', OrdinalEncoder(), ord_cols)
], remainder='passthrough')
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model',XGBRegressor(
    n_estimators=300,
    learning_rate=1,
    max_depth=9,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha = 8,
    reg_lambda = 0,
    random_state=42,
    n_jobs=-1
))])



In [ ]:


pipeline.fit(x_train,y_train)
y_pred=pipeline.predict(x_test)
y_pred=np.expm1(y_pred)

print(r2_score(np.expm1(y_test),y_pred))
print(mean_absolute_error(np.expm1(y_test),y_pred))

0.8409048826695232
0.34863935076379704


In [ ]:
y_pred=pipeline.predict(x_train)
y_pred=np.expm1(y_pred)

print(r2_score(np.expm1(y_train),y_pred))
print(mean_absolute_error(np.expm1(y_train),y_pred))

0.8654358301554874
0.3195465123367715


In [ ]:
!pip install optuna
import optuna

In [ ]:
def  objective(trial):
   n_estimators=trial.suggest_int('n_enstimator',288,288)
   learning_rate=trial.suggest_int('earning_rate',1,1)
   max_depth=trial.suggest_int('max_depth',3,3)
   reg_lambda=trial.suggest_int('reg_lambda',8,8)
   reg_alpha=trial.suggest_int('reg_alpha',0,1)
   num_cols = ['bedRoom', 'built_up_area', 'servant room', 'store room', 'bathroom']

   cat_cols = ['sector', 'agePossession']

   ord_cols = ['balcony', 'luxury_score', 'property_type', 'furnishing_type']

   preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols),
    ('ord', OrdinalEncoder(), ord_cols)
], remainder='passthrough')
   pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model',XGBRegressor(
    n_estimators= n_estimators,
    learning_rate=learning_rate,
    max_depth=max_depth,
    reg_alpha = reg_alpha,
    reg_lambda =reg_lambda,
    random_state=42,
    n_jobs=-1
))])

   score=cross_val_score(pipeline,x_train,y_train,cv=5,scoring="r2").mean()
   return score




In [ ]:
study=optuna.create_study(direction="maximize",sampler=optuna.samplers.TPESampler())
study.optimize(objective,n_trials=50)

[I 2026-04-14 11:30:55,963] A new study created in memory with name: no-name-07981cee-7d0e-4ffd-8367-91b035d40d4a
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros

[I 2026-04-14 11:30:57,772] Trial 0 finished with value: 0.8469979427903244 and parameters: {'n_enstimator': 288, 'earning_rate': 1, 'max_depth': 3, 'reg_lambda': 8, 'reg_alpha': 1}. Best is trial 0 with value: 0.8469979427903244.
/usr/local/lib/python3.12/dist-pac

In [ ]:
print(study.best_trial.value)
print(study.best_trial.params)

0.8625752472201901
{'n_enstimator': 288, 'earning_rate': 1, 'max_depth': 3, 'reg_lambda': 8, 'reg_alpha': 0}


In [ ]:
# {'n_enstimator': 288, 'earning_rate': 1, 'max_depth': 3, 'reg_lambda': 8, 'reg_alpha': 0}



In [ ]:
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [ ]:
plot_param_importances(study).show()